# 02 — Train: U-Net parcel boundary segmentation (Colab)

**Runtime: GPU (T4 minimum; L4 / A100 faster).** Runtime menu → Change runtime type → GPU.

Prereq: `01_prep_colab.ipynb` has run and `/content/drive/MyDrive/aigeolab_train/{tiles, labels, manifest.csv}` exists.

Pipeline (v1.2 — fair loss + IoU only inside annotated mouzas):
1. Install deps + mount Drive + clone repo.
2. Load config; resolve Colab paths.
3. Read manifest; mouza-disjoint train/val split.
4. Rasterise polygons → 3-px boundary mask **AND** a 'valid' mask (filled polygons + dilation) — labels are valid only inside annotated mouza coverage.
5. Patch each 10000×10000 tile into 512×512 windows; drop patches that don't overlap any annotated coverage.
6. Copy tiles + masks to Colab's local SSD.
7. Pre-extract all patches into RAM (eliminates per-batch TIFF I/O).
8. In-memory Dataset + DataLoaders (returns img + mask + valid).
9. Train U-Net with **valid-masked** BCE+Dice loss and val IoU. Predictions outside annotated areas no longer count for or against — fixes the 'model is right, IoU is wrong' problem.
10. Qualitative eval overlay showing RGB, GT boundaries, prediction, and the annotated coverage region.

In [ ]:
# --- Cell 1: install deps + mount Drive + clone repo ---
!pip install -q rasterio shapely segmentation-models-pytorch albumentations opencv-python-headless pyyaml pyshp

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO_URL = 'https://github.com/tahmid013/AIGEOLAB_OFFICE.git'
REPO_DIR = '/content/AIGEOLAB_OFFICE'
if os.path.isdir(REPO_DIR):
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True).stdout)
os.chdir(REPO_DIR)


In [ ]:
# --- Cell 2: load config, resolve Colab paths ---
import yaml
from pathlib import Path

with open('config.yaml') as f: CFG = yaml.safe_load(f)
ENV = 'colab'
P = CFG['paths'][ENV]
STAGE = Path(P['staging_root'])
assert STAGE.is_dir(), f'{STAGE} not found. Run 01_prep_colab.ipynb first.'
TILES   = STAGE / 'tiles'
LABELS  = STAGE / 'labels'
MASKS   = STAGE / 'masks';   MASKS.mkdir(exist_ok=True)
print('staging:', STAGE)
print('manifest exists:', (STAGE / 'manifest.csv').exists())
print('tiles  :', len(list(TILES.glob('*.tif'))))
print('labels :', len(list(LABELS.glob('*.shp'))))


In [ ]:
# --- Cell 3: read manifest, mouza-disjoint train/val split ---
import csv, random
rows = []
with open(STAGE / 'manifest.csv', newline='', encoding='utf-8') as fh:
    for r in csv.DictReader(fh):
        rows.append({'x': int(r['tile_x_km']), 'y': int(r['tile_y_km']),
                     'tif': STAGE / r['tile_tif_relpath'],
                     'mouzas': r['mouza_label_stems'].split('|')})
print(f'manifest rows: {len(rows)}')

all_mouzas = sorted({m for r in rows for m in r['mouzas']})
random.Random(CFG['split']['seed']).shuffle(all_mouzas)
n_val = max(1, int(len(all_mouzas) * CFG['split']['val_mouza_fraction']))
if len(all_mouzas) - n_val < 1:
    n_val = len(all_mouzas) - 1
val_mouzas   = set(all_mouzas[:n_val])
train_mouzas = set(all_mouzas[n_val:])
print(f'Train mouzas: {len(train_mouzas)}  Val mouzas: {len(val_mouzas)}')
print('  val =', sorted(val_mouzas))


In [ ]:
# --- Cell 4: rasterise boundary AND valid masks (cached) ---
# boundary mask: 3-px lines along every plot polygon's exterior (GT target)
# valid    mask: filled polygons + dilation. Marks the region where labels are
#                meaningful. Outside this region the labels are unknown — we
#                shouldn't reward or penalise the model there.
import numpy as np, rasterio, cv2, shapefile
from tqdm.auto import tqdm

THICK = CFG['dataset']['boundary_thickness_px']
VALID_PAD = max(8, THICK + 5)  # dilation to include the boundary band around polygons

def read_polys_pyshp(shp_path):
    rdr = shapefile.Reader(str(shp_path))
    polys = []
    for shp in rdr.shapes():
        pts = shp.points
        parts = list(shp.parts) + [len(pts)]
        for i in range(len(parts) - 1):
            polys.append(pts[parts[i]: parts[i+1]])
    return polys

def boundary_and_valid(tif_path, mouza_stems, thick=THICK, pad=VALID_PAD):
    with rasterio.open(tif_path) as ds:
        H, W = ds.height, ds.width
        T = ds.transform
    boundary = np.zeros((H, W), dtype=np.uint8)
    valid    = np.zeros((H, W), dtype=np.uint8)
    for stem in mouza_stems:
        shp_path = LABELS / f'{stem}.shp'
        if not shp_path.exists(): print(f'  WARN missing label: {shp_path.name}'); continue
        for ring in read_polys_pyshp(shp_path):
            xs = np.array([p[0] for p in ring], dtype=np.float64)
            ys = np.array([p[1] for p in ring], dtype=np.float64)
            cols, rows_ = ~T * (xs, ys)
            pts = np.stack([cols, rows_], axis=1).astype(np.int32)
            cv2.polylines(boundary, [pts.reshape(-1, 1, 2)], isClosed=True, color=255, thickness=thick)
            cv2.fillPoly(valid, [pts], color=255)
    # Dilate the filled polygons so the boundary band sits inside the valid region
    k = np.ones((pad, pad), dtype=np.uint8)
    valid = cv2.dilate(valid, k)
    return boundary, valid

for r in tqdm(rows, desc='rasterising'):
    bp = MASKS / (Path(r['tif']).stem + '_mask.png')
    vp = MASKS / (Path(r['tif']).stem + '_valid.png')
    if bp.exists() and vp.exists() and bp.stat().st_size > 0 and vp.stat().st_size > 0:
        continue
    b, v = boundary_and_valid(r['tif'], r['mouzas'])
    cv2.imwrite(str(bp), b)
    cv2.imwrite(str(vp), v)
print('masks (boundary + valid) ->', MASKS)


In [ ]:
# --- Cell 5: build patch lists (drop patches that don't overlap valid region) ---
PATCH  = CFG['dataset']['patch_size']
STRIDE = CFG['dataset']['stride']

def is_train_tile(r): return all(m in train_mouzas for m in r['mouzas'])
def is_val_tile(r):   return all(m in val_mouzas   for m in r['mouzas'])

train_patches, val_patches = [], []
for r in rows:
    target = train_patches if is_train_tile(r) else (val_patches if is_val_tile(r) else None)
    if target is None: continue
    with rasterio.open(r['tif']) as ds: H, W = ds.height, ds.width
    mp = MASKS / (Path(r['tif']).stem + '_mask.png')
    vp = MASKS / (Path(r['tif']).stem + '_valid.png')
    for top in range(0, H - PATCH + 1, STRIDE):
        for left in range(0, W - PATCH + 1, STRIDE):
            target.append({'tif': str(r['tif']), 'mask': str(mp), 'valid': str(vp),
                           'top': top, 'left': left})

if CFG['dataset']['drop_empty_patches']:
    print('Filtering patches with no valid coverage (per-tile mask cached)...')
    cache = {}
    def get(path):
        if path not in cache: cache[path] = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        return cache[path]
    def has_valid(p):
        v = get(p['valid'])
        return v[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH].any()
    train_patches = [p for p in train_patches if has_valid(p)]
    val_patches   = [p for p in val_patches   if has_valid(p)]

print(f'Train patches: {len(train_patches)}   Val patches: {len(val_patches)}')
assert train_patches, 'No train patches. Stage more mouzas (raise max_mouzas in config).'
assert val_patches,   'No val patches.'


In [ ]:
# --- Cell 6: copy tiles + masks + valid masks to Colab's local SSD ---
import shutil, time
LOCAL = Path('/content/local_train')
LOCAL_TILES = LOCAL / 'tiles'; LOCAL_TILES.mkdir(parents=True, exist_ok=True)
LOCAL_MASKS = LOCAL / 'masks'; LOCAL_MASKS.mkdir(parents=True, exist_ok=True)

unique_tifs = sorted({Path(p['tif']) for p in train_patches + val_patches})
t0 = time.time()
for src in unique_tifs:
    dst = LOCAL_TILES / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print(f'  copying {src.name} ({src.stat().st_size/1e6:.0f} MB) ...')
        shutil.copy2(src, dst)
    for suffix in ('_mask.png', '_valid.png'):
        m_src = MASKS / (src.stem + suffix)
        m_dst = LOCAL_MASKS / m_src.name
        if m_src.exists() and (not m_dst.exists() or m_dst.stat().st_size != m_src.stat().st_size):
            shutil.copy2(m_src, m_dst)

for p in train_patches + val_patches:
    p['tif']   = str(LOCAL_TILES / Path(p['tif']).name)
    p['mask']  = str(LOCAL_MASKS / Path(p['mask']).name)
    p['valid'] = str(LOCAL_MASKS / Path(p['valid']).name)

local_gb = sum(f.stat().st_size for f in LOCAL_TILES.glob('*.tif')) / 1e9
print(f'\nLocal cache: {local_gb:.2f} GB in {time.time()-t0:.0f}s')


In [ ]:
# --- Cell 7: pre-extract ALL patches (img + boundary mask + valid mask) into RAM ---
# Memory cost rule of thumb: N_patches * (3 + 1 + 1) * PATCH^2 bytes.
# 1856 patches @ 512^2 -> ~2.4 GB. Colab gives 12+ GB RAM.

def extract_all(patches, name):
    n = len(patches)
    imgs   = np.empty((n, PATCH, PATCH, 3), dtype=np.uint8)
    masks  = np.empty((n, PATCH, PATCH),    dtype=np.uint8)
    valids = np.empty((n, PATCH, PATCH),    dtype=np.uint8)
    by_tif = {}
    for i, p in enumerate(patches):
        by_tif.setdefault(p['tif'], []).append((i, p))
    for tif_path, items in tqdm(by_tif.items(), desc=f'extract {name}'):
        with rasterio.open(tif_path) as ds:
            tile = ds.read([1, 2, 3])
        tile = np.transpose(tile, (1, 2, 0))
        mask_full  = cv2.imread(items[0][1]['mask'],  cv2.IMREAD_UNCHANGED)
        valid_full = cv2.imread(items[0][1]['valid'], cv2.IMREAD_UNCHANGED)
        for i, p in items:
            imgs[i]   = tile[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH]
            masks[i]  = (mask_full[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH]  > 0).astype(np.uint8)
            valids[i] = (valid_full[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH] > 0).astype(np.uint8)
        del tile, mask_full, valid_full
    return imgs, masks, valids

print(f'Pre-extracting {len(train_patches)} train + {len(val_patches)} val patches...')
train_imgs, train_masks, train_valids = extract_all(train_patches, 'train')
val_imgs,   val_masks,   val_valids   = extract_all(val_patches,   'val')
total_gb = (train_imgs.nbytes + train_masks.nbytes + train_valids.nbytes +
            val_imgs.nbytes + val_masks.nbytes + val_valids.nbytes) / 1e9
print(f'\nCached in RAM: train {train_imgs.shape}  val {val_imgs.shape}   = {total_gb:.2f} GB')
print(f'  train valid coverage: {(train_valids > 0).mean()*100:.1f}% of pixels')
print(f'  val   valid coverage: {(val_valids > 0).mean()*100:.1f}% of pixels')


In [ ]:
# --- Cell 8: in-memory Dataset that yields (img, mask, valid) ---
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def build_aug(train):
    A_CFG = CFG['augment']
    if train:
        return A.Compose([
            A.HorizontalFlip(p=A_CFG['hflip_p']),
            A.VerticalFlip(p=A_CFG['vflip_p']),
            A.RandomRotate90(p=A_CFG['rot90_p']),
            A.RandomBrightnessContrast(brightness_limit=A_CFG['brightness'], contrast_limit=A_CFG['contrast'], p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], additional_targets={'valid': 'mask'})
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'valid': 'mask'})

class InMemoryDataset(Dataset):
    def __init__(self, imgs, masks, valids, train):
        self.imgs, self.masks, self.valids = imgs, masks, valids
        self.aug = build_aug(train)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        out = self.aug(image=self.imgs[i],
                       mask =self.masks[i].astype(np.float32),
                       valid=self.valids[i].astype(np.float32))
        return out['image'], out['mask'].unsqueeze(0), out['valid'].unsqueeze(0)

train_ds = InMemoryDataset(train_imgs, train_masks, train_valids, train=True)
val_ds   = InMemoryDataset(val_imgs,   val_masks,   val_valids,   train=False)
BS = CFG['train']['batch_size']
train_dl = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)
print(f'train batches: {len(train_dl)}   val batches: {len(val_dl)}')


In [ ]:
# --- Cell 9: model + valid-masked train loop ---
import segmentation_models_pytorch as smp
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

M = CFG['model']
model = getattr(smp, M['arch'])(
    encoder_name=M['encoder'], encoder_weights=M['encoder_weights'],
    in_channels=M['in_channels'], classes=M['classes'],
).to(DEVICE)

BCE_W, DICE_W = CFG['train']['loss']['bce_weight'], CFG['train']['loss']['dice_weight']

def masked_bce(logits, y, v):
    per_pix = F.binary_cross_entropy_with_logits(logits, y, reduction='none')
    return (per_pix * v).sum() / v.sum().clamp(min=1.0)

def masked_dice(logits, y, v, eps=1.0):
    p  = torch.sigmoid(logits) * v
    yv = y * v
    inter = (p * yv).sum(dim=(1, 2, 3))
    denom = p.sum(dim=(1, 2, 3)) + yv.sum(dim=(1, 2, 3))
    return (1 - (2 * inter + eps) / (denom + eps)).mean()

def loss_fn(logits, y, v):
    return BCE_W * masked_bce(logits, y, v) + DICE_W * masked_dice(logits, y, v)

def masked_iou(logits, y, v, thr=0.5, eps=1e-7):
    p  = (torch.sigmoid(logits) > thr).float() * v
    yv = y * v
    inter = (p * yv).sum(dim=(1, 2, 3))
    union = ((p + yv) >= 1).float().sum(dim=(1, 2, 3))
    return ((inter + eps) / (union + eps)).mean().item()

opt = torch.optim.AdamW(model.parameters(), lr=CFG['train']['lr'], weight_decay=CFG['train']['weight_decay'])
EPOCHS = CFG['train']['epochs']
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = GradScaler('cuda', enabled=CFG['train']['amp'])

CKPT = STAGE / 'checkpoints'; CKPT.mkdir(exist_ok=True)
best_iou = -1.0

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    model.train(); tr_loss = 0.0
    for x, y, v in train_dl:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        v = v.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=CFG['train']['amp']):
            logits = model(x); loss = loss_fn(logits, y, v)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tr_loss += loss.item() * x.size(0)
    tr_loss /= max(1, len(train_ds))

    model.eval(); v_loss, v_iou, n = 0.0, 0.0, 0
    with torch.no_grad():
        for x, y, v in val_dl:
            x, y, v = x.to(DEVICE), y.to(DEVICE), v.to(DEVICE)
            logits = model(x); loss = loss_fn(logits, y, v)
            v_loss += loss.item() * x.size(0)
            v_iou  += masked_iou(logits, y, v) * x.size(0)
            n      += x.size(0)
    v_loss /= max(1, n); v_iou /= max(1, n)
    sched.step()

    dt = time.time() - t_ep
    print(f'ep {epoch:02d}/{EPOCHS}  train_loss={tr_loss:.4f}  val_loss={v_loss:.4f}  val_iou(masked)={v_iou:.4f}  lr={opt.param_groups[0]["lr"]:.2e}  {dt:.0f}s')
    if v_iou > best_iou:
        best_iou = v_iou
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_iou': v_iou, 'cfg': CFG}, CKPT / 'best.pt')
        print(f'  saved best (val_iou={v_iou:.4f})')

print(f'\nDone. best val IoU (masked) = {best_iou:.4f}')


In [ ]:
# --- Cell 10: qualitative eval on a held-out val tile ---
import matplotlib.pyplot as plt

val_tile_rows = [r for r in rows if all(m in val_mouzas for m in r['mouzas'])]
assert val_tile_rows, 'No val tile found.'
vr = val_tile_rows[0]
vr_tif   = LOCAL_TILES / Path(vr['tif']).name if (LOCAL_TILES / Path(vr['tif']).name).exists() else vr['tif']
vr_mask  = LOCAL_MASKS / (Path(vr['tif']).stem + '_mask.png')  if (LOCAL_MASKS / (Path(vr['tif']).stem + '_mask.png')).exists()  else (MASKS / (Path(vr['tif']).stem + '_mask.png'))
vr_valid = LOCAL_MASKS / (Path(vr['tif']).stem + '_valid.png') if (LOCAL_MASKS / (Path(vr['tif']).stem + '_valid.png')).exists() else (MASKS / (Path(vr['tif']).stem + '_valid.png'))
print('Visualising:', Path(vr_tif).name, '  mouzas:', vr['mouzas'])

with rasterio.open(vr_tif) as ds:
    H, W = ds.height, ds.width
    top, left = H // 2 - 512, W // 2 - 512
    crop = ds.read([1, 2, 3], window=rasterio.windows.Window(left, top, 1024, 1024))
crop = np.transpose(crop, (1, 2, 0))

model.eval()
norm = A.Compose([A.Normalize(), ToTensorV2()])
with torch.no_grad():
    pred = np.zeros((1024, 1024), dtype=np.float32)
    cnt  = np.zeros((1024, 1024), dtype=np.float32)
    for top2 in range(0, 1024 - PATCH + 1, STRIDE):
        for left2 in range(0, 1024 - PATCH + 1, STRIDE):
            sub = crop[top2:top2+PATCH, left2:left2+PATCH]
            x = norm(image=sub)['image'].unsqueeze(0).to(DEVICE)
            p = torch.sigmoid(model(x))[0, 0].cpu().numpy()
            pred[top2:top2+PATCH, left2:left2+PATCH] += p
            cnt[top2:top2+PATCH, left2:left2+PATCH]  += 1
    pred = pred / np.maximum(cnt, 1)

gt_full  = cv2.imread(str(vr_mask),  cv2.IMREAD_UNCHANGED)
val_full = cv2.imread(str(vr_valid), cv2.IMREAD_UNCHANGED)
gt    = gt_full [top:top+1024, left:left+1024]
valid = val_full[top:top+1024, left:left+1024]

# Two IoUs over the crop, at threshold 0.5: unmasked (penalised by sparse GT) vs masked.
p_bin = (pred > 0.5).astype(np.uint8) * 255
g_bin = (gt   > 0).astype(np.uint8)   * 255
v_bin = (valid > 0).astype(np.uint8)
inter_u = ((p_bin > 0) & (g_bin > 0)).sum();  union_u = ((p_bin > 0) | (g_bin > 0)).sum()
inter_m = ((p_bin > 0) & (g_bin > 0) & (v_bin > 0)).sum();  union_m = (((p_bin > 0) | (g_bin > 0)) & (v_bin > 0)).sum()
iou_u = inter_u / max(1, union_u);  iou_m = inter_m / max(1, union_m)
print(f'crop IoU unmasked = {iou_u:.4f}   crop IoU masked-to-valid = {iou_m:.4f}')

fig, ax = plt.subplots(1, 4, figsize=(24, 6))
ax[0].imshow(crop); ax[0].set_title('RGB')
ax[1].imshow(crop); ax[1].imshow(gt,    cmap='Reds',   alpha=0.5); ax[1].set_title('Ground-truth boundaries (red)')
ax[2].imshow(crop); ax[2].imshow(pred,  cmap='Blues',  alpha=0.6); ax[2].set_title(f'Predicted boundary  (IoU unmasked {iou_u:.2f})')
ax[3].imshow(crop); ax[3].imshow(valid, cmap='Greens', alpha=0.3); ax[3].imshow(pred * (valid > 0), cmap='Blues', alpha=0.6); ax[3].set_title(f'Pred masked to coverage (IoU {iou_m:.2f})')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
